# LSTM Gates — Forget, Input, Output

LSTMs use gates to control what information flows through time.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
torch.manual_seed(42); np.random.seed(42)


In [ ]:
class DummyDataGenerator:
    def __init__(self, seq_len=30):
        self.x = torch.randn(1, seq_len, 4)
    def input(self): return self.x

class SequenceDataset(Dataset):
    def __init__(self, x): self.x = x
    def __len__(self): return 1
    def __getitem__(self, i): return self.x[i]

class LSTMModel(nn.Module):
    def __init__(self, d_in=4, hidden=16):
        super().__init__()
        self.lstm = nn.LSTM(d_in, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, (h, c) = self.lstm(x)
        return self.fc(out[:, -1]), out

x = DummyDataGenerator().input()
model = LSTMModel()
pred, all_h = model(x)

fig, ax = plt.subplots(figsize=(10, 4))
for i in range(min(4, all_h.shape[-1])):
    ax.plot(all_h[0, :, i].detach().numpy(), label=f'hidden dim {i}', alpha=0.8)
ax.set_title('LSTM hidden state evolution over time'); ax.legend(); ax.set_xlabel('time step')
plt.show()

# Gate concept diagram (schematic)
fig, ax = plt.subplots(figsize=(8, 3))
gates = ['Forget gate', 'Input gate', 'Output gate', 'Cell state']
vals = [0.2, 0.7, 0.5, 1.0]
ax.barh(gates, vals, color=['#e74c3c','#2ecc71','#3498db','#9b59b6'])
ax.set_xlim(0, 1); ax.set_title('LSTM gates (example values at one step — σ(output) ∈ [0,1])')
plt.tight_layout(); plt.show()